In [12]:
import sys
from pathlib import Path
import json
import requests


print(
    f"Python Version: {sys.version_info.major}.{sys.version_info.minor}.{sys.version_info.micro}"
)
print(f"Environment: {sys.executable}")

# Find project root and add to Python path
current_dir = Path.cwd()
if current_dir.name == "notebooks":
    project_root = current_dir.parent
elif (current_dir / "compose.yml").exists():
    project_root = current_dir
else:
    project_root = None

if project_root and (project_root / "compose.yml").exists():
    print(f"Project root: {project_root}")
    sys.path.insert(0, str(project_root))
else:
    print("Missing compose.yml - check directory")
    exit()

Python Version: 3.12.12
Environment: /Users/surekha/Documents/projects/RAG/research-assistant/.venv/bin/python
Project root: /Users/surekha/Documents/projects/RAG/research-assistant


In [13]:
from src.services.opensearch.factory import make_opensearch_client
from opensearchpy import OpenSearch

opensearch_client = make_opensearch_client()
opensearch_client.host = "http://localhost:9200"
opensearch_client.client = OpenSearch(hosts=["http://localhost:9200"],http_compress=True,use_ssl=False,verify_cert=False,ssl_assert_hostname=False,ssl_show_warn=False)

print(f"Client configured with host: {opensearch_client.host}")
print(f"Index name: {opensearch_client.index_name}")

is_healthy = opensearch_client.health_check()
if is_healthy:
    print("OpenSearch health check: PASSED")
else:
    print("OpenSearch health check: FAILED")

Client configured with host: http://localhost:9200
Index name: arxiv-papers
OpenSearch health check: PASSED


In [14]:
# Display Index configuration

from src.services.opensearch.index_config import ARXIV_PAPERS_INDEX,ARXIV_PAPERS_MAPPING

print(f"Index Name: {ARXIV_PAPERS_INDEX}")

properties = ARXIV_PAPERS_MAPPING["mappings"]["properties"]

for field_name,config in properties.items():
    field_type = config.get("type")
    analyzer = config.get("analyzer","")
    if analyzer:
        print(f"    . {field_name}: {field_type} [{analyzer}]")
    else:
        print(f"    . {field_name}: {field_type}")

Index Name: arxiv-papers
    . arxiv_id: keyword
    . title: text [text_analyzer]
    . authors: text [standard_analyzer]
    . abstract: text [text_analyzer]
    . categories: keyword
    . raw_text: text [text_analyzer]
    . pdf_url: keyword
    . published_date: date
    . created_at: date
    . updated_at: date


In [15]:
try:
    index_exists = opensearch_client.client.indices.exists(index=opensearch_client.index_name)

    if index_exists:
        print(f" Index {opensearch_client.index_name} already exists")

        stats = opensearch_client.get_index_stats()
        if stats and 'error' not in stats:
            print(f"    Document: {stats.get('document_count',0)}")
            print(f"    Size:{stats.get('size_in_bytes',0):,} bytes")
    else:
        print(f"Creating new index: {opensearch_client.index_name}")
        success = opensearch_client.create_index()

        if success:
            print(f"    Index created successfully")
        else:
            print(f"    Index creation failed")

except Exception as e:
    print(f"    Error with index management")

 Index arxiv-papers already exists
    Document: 0
    Size:208 bytes


In [ ]:
stats = opensearch_client.get_index_stats()

if stats and 'error' not in stats:
    doc_count = stats.get('document_count',0)

    if doc_count>0:
        print("  Success! Found {doc_count} documents in OpenSearch")

        sample = opensearch_client.search_papers("*", size=3)
        if sample.get("hits"):
            print("\n Sample papers")

            for i, paper in enumerate(sample['hits'],1):
                title = paper.get('title', 'Unknown')[:60]
                print(f"    {i}. {title}...")
    else:
        print(" No documents in OpenSearch yet")
        print("\nPlease run the Airflow DAG first (see instructions above)")
else:
    print("Could not retrieve index stats")